# ECDSA/DSA с Nonce со Статистическим Отклонением

**NB!** Эту записную книжку надо открывать при помощи sagemath в режиме jupyter, а не с использованием обычного python, так как ему необходимы API sagemath.
Установите [Sagemath](https://www.sagemath.org/) и запустите:
```sh
sage -n jupyter
```
Мы прошли, что имплементации DSA и ECDSA становятся уязвимыми, если:

1. Использовать предсказуемые $k$ (nonce)

2. Использовать повторяющиеся $k$ 

Также я упоминал, что использование $k$ со статистическим отклонением также может привести к раскрытию ключа. Это упражнение сфокусировано как раз на этой проблеме.

## Введение 

Сначала, давайте вспомним, из чего состоят DSA и ECDSA. В случае DSA  у нас есть хеш-функция $Hash(m)$, мультипликативнaя группа $Z^*_p$ по модулю простого числа $p$ и генератор $g$ порядка $q$, где $q$ - самый большой простой делитель $p-1$ (т.е. $g$ - генератор самой большой подгруппы с порядком, равным простому числу, группы $Z^*_p$). Закрытый ключ - $x\ \in(0,q)$, а $y=g^x\  mod\  p$  открытый ключ.

Алгоритм подписи $m$:

1. Выбрать случайный nonce (number used once; число, используемое один раз) $k\in (0,q)$

2. Вычислить $r = (g^k\  mod\  p)\  mod\  q$. Если $r=0$, то вернуться к пункту 1.

3. Вычислить $s = \frac{\mathit{Hash}(m)+x\cdot r}{k}\  mod\  q$. Если $s=0$, то вернуться к пункту 1.

4. Вернуть $(r,s)$. Это и есть подпись

Чтобы проверить подпись, нам надо удостовериться, что $r,s \in (0,q)$ и следующее уравнение верно:
$$(g^\frac{Hash(m)}{s}\cdot y^\frac{r}{s}\  mod p)\  mod \  q=r$$

В случае ECDSA дана эллиптическая кривая $E(\mathit{GF}(p))$, некоторая точка $G$ на кривой в качестве генератора подгруппы простого порядка $q$, т.е. $q\cdot G=0$ (Точка на Бесконечности). Также определена хеш-функция $\mathit{Hash}(m)$. Чтобы создать ключи мы:

1. Выбираем случайное $d \in (0,q)$ (закрытый ключ)

2. Вычисляем точку $Q=d\cdot G$ (открытый ключ)

Алгоритм подписи $m$:

1. Выбрать случайный nonce (number used once; число, используемое один раз) $k\in (0,q)$

2. Вычислить $(x_1,y_1)=k\cdot G$

3. Вычислить $r=x_1\  mod \  q$. Если $r=0$, то вернуться к пункту 1.

4. Вычислить $s=\frac{\mathit{Hash}(m)+r\cdot d}{k}\  mod\  q$. Если $s=0$, то вернуться к пункту 1.

5. Вернуть $(r,s)$. Это и есть подпись.

Алгоритм проверки подписи:

1. Проверить, что $r,s \in (0,q)$

2. Вычислить $(x_1,y_1)=\frac{\mathit{Hash}(m)}{s}\cdot G+\frac{r}{s}\cdot Q$

3. Проверить, что $r=x_1 \  mod \  q$

В этот раз я пропустил доказательство корректности, раз уж мы его уже прошли в прошлом задании, а в атаке оно не поможет.

Так что же мы подразумеваем под nonce со статистическим отклонением? Это значит, что распределение получаемых nonce сосредоточено в каком-то регионе. Например, 8 верхних битов $k$ всегда равны 0. Так как $s = \frac{\mathit{Hash}(m)+x\cdot r}{k}\  mod\  q$, то

$$s\cdot k = \mathit{Hash}(m)+x\cdot r\  mod \  q$$

$$k = \frac{\mathit{Hash}(m) + x\cdot r}{s}\  mod \  q$$

$$x\cdot \frac{-r}{s}=\frac{\mathit{Hash}(m)}{s} - k \  mod \  q$$

Мы можем представить это выражение в следующем виде:

$x\cdot a=b-c\  mod \  q$, where $a=\frac{-r}{s}\  mod \  q$, $b=\frac{\mathit{Hash}(m)}{s}$ и $c=k$. Заметим, что $c$ намного меньше  $x$, $a$ и $b$, и мы можем сказать, что $x\cdot a \simeq b \  mod\  q$.

## HNP, CVP, LLL и Gram-Schmidt (Грам-Шмидт)

То, на что мы смотрим - это вариант проблемы скрытого числа (the Hidden Number Problem, HNP):

Восстановление $\alpha \in F_p$ такого, что для многих известных случайных $t \in Fp$ нам известны $MSB_{l,p}(\alpha t)$ для некоторого $l>0$. Алгоритм для решения этой задачи основывается на проблеме ближайшего вектора (CVP, Closest Vector Problem). Но чтобы описать её, нам надо сначала познакомиться с решётками (lattices).

Пусть $\{b_1,...,b_s\}$ - это множество линейно независимых векторов в $\mathbb{R}^s$. 

Множество векторов $$L=\{z | z=\sum_{i=1}^{s} c_ib_i,\  c_1,...,c_s \in \mathbb{Z}\}$$  называется полноранговой решёткой размерности $s$. Множество $\{b_1,...,b_s\}$ называется базисом $L$.

The Closest Vector Problem (CVP, Проблема Ближайшего Вектора):

При данном векторе $r \in \mathbb{R}^s$ найти вектор решётки $v \in L$ такой, что
$$ ||r-v||=min_{z\in L}||r-z|| $$
CVP - это NP-полная задача, но существуют полиномиальные алгоритмы, которые позволяют найти примерное решение, например, LLL (Lenstra, Lenstra, Lovasz; алгоритм Ленстры - Ленстры - Ловаса).

LLL использует процесс Грама-Шмидта (Gram-Schmidt process). Это метод ортонормирования набора векторов в пространстве с определенным скалярным умножением.

Даны $u,v \in \mathbb{R}^s$, оператор проекции определяется как $proj_u(v)=\frac{(u,v)}{(u,u)}u$, где $(u,v)$ обозначает скалярное произведение. Для указанного набора векторов $v_1,...,v_k \in \mathbb{R}^s$ в Граме-Шмидте сначала выполняется ортогонализация:

$$ u_1= v_1 $$
$$ u_2= v_2- proj_{u_1}(v_2)$$
$$...$$
$$ u_k=v_k-\sum_{j=1}^{k-1}proj_{u_j}(v_k) $$

Цель - получить отогональный базис (когда скалярное произведение каждой пары различных векторов базиса равно нулю). Далее каждый вектор надо поделить на корень скалярного произведения вектора на самого себя, чтобы получить ортонормированный базис.

Алгоритм LLL вычисляет LLL-редуцированные базисы. Если дан базис
$B = \{b_1, b_2,...,b_n\}$, ортогональный базис, полученный от него процессом Грама-Шмидта (без нормирования)
$$B^{*}=\{b_1^{*}, b_2^{*},...,b_n^{*}\}$$
и коэффициенты Грама-Шмидта

$\mu_{i,j}=\frac{(b_i,b_j^{*})}{(b_j^{*},b_j^{*})}$, для всех $1 \le j < i \le n$.

Тогда базиc $B$ LLL-редуцирован, если существует параметр $\delta \in (0.25, 1 {]}$ такой, что выполняются следующие условия:

1. (Условие уменьшения размера) For $1\le j < i \le n : |\mu_{i,j}\le 0.5|$. Это свойство гарантирует уменьшение длины векторов у упорядоченного базиса.

2. (Условие Ловаса) Для $k=2,3,...,n: \delta||b_{k-1}^{*}||^2 \le ||b_{k}^{*}||^2+\mu_{k,k-1}^{2}||b_{k-1}^{*}||^2$.

По сути, $\delta$ указывает, насколько хорошо редуцирован базис. Ниже приведен псевдокод алгоритма LLL на питоноподобном синтаксе. $B$ - это матрица, каждый ряд которой представляет отдельный вектор базиса.

```python
def LLL(B, delta):
    B = copy(B)
    Q = gramschmidt(B)
    def mu(i, j):
        v = B[i]
        u = Q[j]
        return (v*u) / (u*u)

    n = len(B)
    k = 1

    while k < n:
        for j in reverse(range(k)):
            if abs(mu(k, j)) > 1/2:
                B[k] := B[k] - round(mu(k, j))*B[j]
                Q := gramschmidt(B)

        if (Q[k]*Q[k]) >= (delta - mu(k, k-1)^2) * (Q[k-1]*Q[k-1]):
            k := k + 1
        else:
            B[k], B[k-1] := B[k-1], B[k]
            Q := gramschmidt(B)
            k := max(k-1, 1)

    return B
```

Больше про LLL можно прочитать на [википедии](https://ru.wikipedia.org/wiki/%D0%90%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC_%D0%9B%D0%B5%D0%BD%D1%81%D1%82%D1%80%D1%8B_%E2%80%94_%D0%9B%D0%B5%D0%BD%D1%81%D1%82%D1%80%D1%8B_%E2%80%94_%D0%9B%D0%BE%D0%B2%D0%B0%D1%81%D0%B0). Основная идея в том, что он пытается редуцировать базис, переставляя и вычитая векторы друг из друга. В результате каждый вектор в новом базисе будет линейной комбинацией векторов в изначальном базисе.

## Как решать проблему со статистическим отклонением nonce у DSA/ECDSA при помощи LLL

Вспомним, что $x\cdot a \simeq b \  mod\ q$, что значит, что $x\cdot a \simeq b + j \cdot q, j \in \mathbb{Z}$. Нам нужно найти $x$. $a$, $b$ и $q$ известны. Поэтому для нескольких элементов $a_1, a_2,..., a_n$, $b_1, b_2,..., b_n$ мы можем построить следующую матрицу:
$$
\begin{vmatrix}
q & 0 & 0 & ... & 0\\
0 & q & 0 & ... & 0\\
0 & 0 & q & ... & 0\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
0 & 0 & 0 & ... & q\\
a_1 & a_2 & a_3 & ... & a_s\\
b_1 & b_2 & b_3 & ... & b_s
\end{vmatrix}
$$

Каждый ряд представляет собой вектор. Нам известно, что $x\cdot \vec{a} \simeq \vec{b}$ (здесь коэффициенты в $Fq$), поэтому один из векторов базиса, который вычислит LLL, будет вектор $\vec{c}=\vec{b}-x\cdot \vec{a}+\sum_{i=1}^{n}(t_i\vec{q_i}), t_i \in \mathbb{Z}$ (здесь коэффициенты в $\mathbb{Q}$). Но в итоге мы хотим вычислить $x$, как нам это сделать? Всё просто, дополним векторы двумя элементами. К векторам, которые содержат $q$ добавляем два нулевых элемента. Мы хотим сбалансировать финальный вектор так, чтобы все элементы в нем были примерно одного порядка.  Если отклонение в верхних $l$ битах $k$, то в векторе $\vec{a}$ мы можем выставить $(n+1)$-й в $\frac{1}{2^l} \in \mathbb{Q}$, а у $\vec{b}$ можно выставить $(n+2)$-й элемент в $\frac{q}{2^l}$. Поскольку конечный вектор будет $\vec{b}-x\cdot \vec{a}$, то его последний элемент будет равен $\frac{q}{2^l}$, а предпоследний - $\frac{x}{2^l}$ поэтому все элементы будут примерно одного порядка.
$$
\begin{vmatrix}
q & 0 & 0 & ... & 0 & 0 & 0\\
0 & q & 0 & ... & 0 & 0 & 0\\
0 & 0 & q & ... & 0 & 0 & 0\\
\vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots\\
0 & 0 & 0 & ... & q & 0 & 0\\
a_1 & a_2 & a_3 & ... & a_n & \frac{1}{2^l} & 0\\
b_1 & b_2 & b_3 & ... & b_n & 0 & \frac{q}{2^l}\\
\end{vmatrix}
$$

Таким образом в конце мы можем проверить все строки (векторы нового базиса) на то, не равен ли последний элемент $\frac{q}{2^l}$, и если обнаружим такой вектор, то предпоследний элемент с высокой вероятностью будет равен $\frac{-x}{2^l}$, из чего мы можем легко вычислить $x$.

Как мы видим, атака состоит из 4х шагов:

1. Собрать достаточное количество подписей $(r,s)$ (оно зависит от отклонения)

2. Вычислить векторы $\vec{a}$ и $\vec{b}$ и определить решётку (матрицу)

3. Использовать LLL для решения SVP (нахождения самого короткого вектора)

4. Вычислить $x$ из полученного базиса

Использовать LLL просто:

In [1]:
from sage.all import *

# Рациональные числа можно использовать так:
print (Rational(1)/2)
M=matrix([[0,1,1],[2,3,1],[1,1,1]])
print (M.LLL())

1/2
[ 1  0  0]
[ 0  1  1]
[ 0  1 -1]


## Назад к отклонениям

Что произойдет в случае, если статистическое отклонение затрагивает нижние биты $k$? Например, нижние $l$ битов $k$ содержат значение $z$: $k=2^l\cdot k_0+z$.

$$s=\frac{\mathit{Hash}(m)+x\cdot r}{k} \  mod \  q$$

$${k}=\frac{\mathit{Hash}(m)+x\cdot r}{s} \  mod \  q$$

$$2^l\cdot k_0+z=\frac{\mathit{Hash}(m)+x\cdot r}{s} \  mod \  q$$

$$2^l\cdot k_0=(\frac{\mathit{Hash}(m)}{s}-z)+\frac{x\cdot r}{s} \  mod \  q$$

$$k_0= \frac{(\frac{\mathit{Hash}(m)}{s}-z)}{2^l} + \frac{x\cdot r}{s\cdot 2^l} \  mod \  q $$

$$x\cdot \frac{-r}{s\cdot 2^l}=\frac{(\frac{\mathit{Hash}(m)}{s}-z)}{2^l}-k_0 \  mod \  q $$

Снова получаем $a\cdot x=b-c$, где $c=k_0$ много меньше чем и $a=\frac{-r}{s\cdot 2^l}$ и $b=\frac{(\frac{H(m)}{s}-z)}{2^l}$. Поэтому мы можем использовать один и тот же метод.

## Задание

Сервер генерирует два ключа. Он использует первый ключ для подписи 80 сообщений вариантом алгоритма со статистическим отклонением в верхних битах, а второй для подписи 80 сообщений алгоритмом с отклонением в нижних битах. Этого количества подписей должно быть более чем достаточно, чтобы восстановить ключи (степень отклонения можете посмотреть в коде ниже). Ваша цель - восстановить ключи и подписать сообщения 'give' и 'flag' как и в предыдущем задании. Рекомендую сначала написать функцию для решения проблемы для обобщенных $\vec{a}$ и $\vec{b}$, а потом написать преобразование $\mathit{Hash}(m),r,s$ в $a, b$. Так Вы сможете переиспользовать код. Также не забывайте, что Вы можете симулировать свой сервер для тестирования (и менять там количество подписей, степень отклонения и другие параметры). 

Удачи!

In [67]:
import random
import time
from Crypto.Util.number import bytes_to_long
import hashlib
# Это те же классы, которые использует сервер
class DSA:
    def __init__(self,g,p,q):
        self.p=p
        self.g=g
        self.q=q
        import random
        self.x=random.randint(1,q-1)
        self.y=pow(g,self.x,self.p)
    
    def get_k(self):
        raise NotImplementedError()
    @staticmethod
    def message_to_hashnum(m):
        if not isinstance(m,bytes):
            raise TypeError()
        return bytes_to_long(hashlib.sha256(m).digest())
    def sign(self,m):
        r=0
        s=0
        while r==0 or s==0:
            k=0
            while k==0:
                k=self.get_k()
            r=pow(g,k,self.p)%self.q
            s=(((DSA.message_to_hashnum(m)+(self.x*r)%self.q)%self.q)*pow(k,q-2,self.q))%self.q
        return (r,s)
    def verify(self,m,r,s,otherY):
        if r==0 or s==0: return False
        w=pow(s,self.q-2,self.q)
        u1=(DSA.message_to_hashnum(m)*w)%self.q
        u2=(r*w)%self.q
        res=(pow(self.g,u1,self.p)*pow(otherY,u2,self.p)%p)%q
        return res==r

    
class MSBBiasedDSA(DSA):
    def get_k(self):
        return random.randint(0,self.q>>32) # first 32 bits -> zeros

class LSBBiased_DSA(DSA):
    def get_k(self):
        return random.randint(0,self.q-1)|((1<<32)-1) # last 32 bits -> zeros





In [68]:
import socket
import re
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1351))
        
    def recv_until(self,symb=b'\n>'):
        """Получаем сообщения от сервера, по умолчанию до следующего приглашения"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def getChallenge(self,show=True):
        print ("Пожалуйста, подождите. Создание подписей может занять некоторое время...")
        data=self.recv_until()
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования юникода. Попробуйте переподключиться к серверу')
            return (None,None)
        if show:
            print (data)
        p=int(re.search(r'(?<=p=)\d+',data).group(0))
        q=(p-1)//2
        g=int(re.search(r'(?<=g=)\d+',data).group(0))
        y1=int(re.search(r'(?<=y1=)\d+',data).group(0))
        y2=int(re.search(r'(?<=y2=)\d+',data).group(0))
        all_tuples=re.findall(r'(?<=\(m,r,s\)=)\([a-z]+,\d+,\d+\)',data)
        top_biased=[]
        for i in range(0,80):
            spl=all_tuples[i][1:-1].split(',')
            top_biased.append((spl[0].encode(),int(spl[1]),int(spl[2])))
        
        low_biased=[]
        for i in range(80,160):
            spl=all_tuples[i][1:-1].split(',')
            low_biased.append((spl[0].encode(),int(spl[1]),int(spl[2])))
        print ('Параметры задания получены')
        return (p,q,g,y1,y2,top_biased,low_biased)
    
    def checkSolution(self,rg,sg,rf,sf, show=True):
        self.s.sendall((str(rg)+' '+str(sg)+' '+str(rf)+' '+str(sf)+'\n').encode())
        data=self.recv_until(b'\n')
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка декодирования юникода. Попробуйте переподключиться к серверу')
            return None
        if show:
            print (data)
        if data.find('flag')!=-1:
            return True
        else:
            data=self.recv_until(b'>')
            try:
                data=data.decode()
            except UnicodeDecodeError:
                print ('Ошибка декодирования юникода. Попробуйте переподключиться к серверу')
                return None
            if show:
                print (data)
            return False
    def __del__(self):
        self.s.close()

vs=VulnServerClient()
(p,q,g,y1,y2,msb_biased,lsb_biased)=vs.getChallenge(show=False)

Пожалуйста, подождите. Создание подписей может занять некоторое время...
Параметры задания получены


In [69]:
bias = 32 # Посмотрите в коде (random.randint(0,self.q>>32))

print(f"p bit_length: {p.bit_length()}")
print(f"q bit_length: {q.bit_length()}")
print(f"generator g: {g}")
print(f"y1 bit_length: {y1.bit_length()}")
print(f"y2 bit_length: {y2.bit_length()}")
print(f"num msb_biased: {len(msb_biased)}")
print(f"num lsb_biased: {len(lsb_biased)}")

# Example
print(f"lsb_biased: {lsb_biased[0]}")

p bit_length: 2048
q bit_length: 2047
generator g: 4
y1 bit_length: 2048
y2 bit_length: 2047
num msb_biased: 80
num lsb_biased: 80
lsb_biased: (b'wickedcool', 1884995604032711681283227632015621611111008933312860209305934142735289080833917918482019121515006380718000141682475514205259996541571002740236884828665860594155874142353732685414597745297745852769371604458568138240613076441139118601792714546294620073044895738711543471740325770541426391114147687417762936812648940384129014580641750650001774741795438566027774668580156761019536639272789431039046710664109864104696477527430773292558382947668957275039897779083670222514202102099648602583121683371649937940868208053022866435228748504323636567998664548667157451390346924423689169605408839952156928680670260192067955701389, 9530338624880667919639373985394647571099319128584478490907451243988694876686948172187062861201167572188061574488530055780194798236839262017731487420626184894178349000223886643408743334042027319560677667386279555156947860329

Подписи в формате `message, r, s.` Всего получено 80 подписей

In [70]:
a_vector=[]
b_vector=[]

# Преобразуйте (H(m),r,s)-> (a,b) для случая MSB (отклонение в верхних битах)
H = DSA.message_to_hashnum 

# Transforming functions
def transform_to_a_b_msb(msb_data, q):
    a_b = []
    for signature in msb_data:
        message, r, s = signature
        hashed_message = H(message)
        s_inv = pow(s, q-2, q)
        a = (-1) * r * s_inv % q
        b = hashed_message * s_inv % q
        a_b.append((a, b))

    return a_b

def transform_to_a_b_lsb(lsb_data, q):
    a_b = []
    for signature in lsb_data:
        message, r, s = signature
        hashed_message = H(message)
        s_inv = pow(s, q-2, q)
        inv_2pow32 = pow(2**32, q-2, q)
        z = (1 << 32) - 1
        a = (-1) * r * s_inv * inv_2pow32 % q
        b = (hashed_message * s_inv % q - z) * inv_2pow32 % q
        a_b.append((a, b))
    return a_b

a_b_pairs_msb = transform_to_a_b_msb(msb_data=msb_biased, q=q)
a_b_pairs_lsb = transform_to_a_b_lsb(lsb_data=lsb_biased, q=q)

print(f"len a_b: {len(a_b_pairs_msb)}, {len(a_b_pairs_lsb)}")

len a_b: 80, 80


Строим базисы решёток

In [71]:
def build_lattice_matrix(a_b_pairs, bias=32):

    n = len(a_b_pairs)
    scale = 2**bias

    M = matrix(QQ, n+2, n+2)

    for i in range(n):
        M[i, i] = q
    
    for i in range(n):
        M[n, i] = a_b_pairs[i][0]
    
    M[n, n] = QQ(1) / scale
    
    M[n, n+1] = 0
    
    for i in range(n):
        M[n+1, i] = a_b_pairs[i][1]
        
    M[n+1, n] = 0
    
    M[n+1, n+1] = QQ(q) / scale  

    return M

L_matrix_msb = build_lattice_matrix(a_b_pairs_msb)
L_matrix_lsb = build_lattice_matrix(a_b_pairs_lsb)

Целевой вектор

In [72]:
r = QQ(q) / 2**bias

LLL-редукция

In [73]:
L_reduced_msb = L_matrix_msb.LLL()
L_reduced_lsb = L_matrix_lsb.LLL()

Ищем ближайший вектор

In [74]:
def extract_key(L_reduced, q, bias=32):
    target_last = QQ(q) / 2**bias
    for row in L_reduced:
        if abs(row[-1]) == target_last:
            x = int(-row[-2] * 2**bias) % q
            return x
    return None

x1 = extract_key(L_reduced_msb, q)
x2 = extract_key(L_reduced_lsb, q)

Получили `x1` и `x2`

In [75]:
print(f"x1: {x1}")

x1: 4458676708544559119486425747130729053599862817899037795979220732711680699770246874858618249963970940463901069990974093001606693689421834077168299335070586828183611282990925710756262522795934821269965015968915085416406287964674696501032328026067342148859695883903696497427463923526121966104450017945624890929965578704481877443337579096607537226756278638326310386561787640328781155904170016285418688713115190622283626101480385522972631467271929922593889746742500932109315846559926011555075330638629564104127869710941483900781854133875811607978436379163297121858334276619834691526180879300526687857454372355849239685673


In [76]:
print(f"x2: {x2}")

x2: 2159720459307696135808181308390618646981764447207276713970270331052227386599069734804127974730999756860322094295239513149588758933090107587950399520931835743711007688162516129326004917220287496882791359276083494761517146940210522052223044419262617202539195049951051848202110458906182374562146493982115099575616932360175317012750565890224520559470167572111640040145093913531201556126723217515473200534784065946248596225308218038328422681794430521664577244705180249655654112201752669236943924094953645667684518631506636335982372854594199378936254761294623362584683514590535418432375281620847685031594363880126384173667


Проверка

In [77]:
if pow(g, x1, p) == y1:
    print(f"MSB key is valid!")
else:
    print(f"MSB key is invalid...")

if pow(g, x2, p) == y2:
    print(f"MSB key is valid!")
else:
    print(f"MSB key is invalid...")

MSB key is valid!
MSB key is valid!


Подписываем

In [78]:
lDSA=LSBBiased_DSA(g,p,q)
lDSA.x=x1
lDSA.y=y1
(r_g,s_g)=lDSA.sign(b'give')
lDSA.x=x2
lDSA.y=y2
(r_f,s_f)=lDSA.sign(b'flag')


In [79]:
vs.checkSolution(r_g, s_g, r_f, s_f)

Congratulations, your flag is: CRYPTOTRAINING{y0u_h4v3_t0_b3_r34lly_c4r3fu11_w1th_th0s3_n0nc3s}.



True